# Fundamental Diagram

The fundamental diagram, the relation between pedestrian density and speed (or flow), is one of the most important characteristics of pedestrian dynamics, as it can be used to e.g., rate the capacity of pedestrian facilities.

This notebook shows a complete end-to-end analysis with *PedPy*:
It computes the fundamental diagrams of a series of uni-directional corridor experiments with the four different measurement methods (Method A-D) described in [Zhang et al. (2011)](https://doi.org/10.1088/1742-5468/2011/06/P06004) and compares the results with the ones reported in the paper.
The methods differ in how and where density and speed are measured, hence they lead to slightly different results.

This guide is designed as a [Jupyter notebook](https://jupyter.org/) which can be downloaded {download}`here <./fundamental_diagram.ipynb>` and run locally.


## Setup

Use the steady state frames, used in the paper describing the methods. Also uses roughly the same style in the plots.

In [ ]:
steady_states = {
    "uo-050-180-180": [211, 800],
    "uo-060-180-180": [243, 771],
    "uo-070-180-180": [203, 1113],
    "uo-100-180-180": [200, 790],
    "uo-145-180-180": [300, 1097],
    "uo-180-180-070": [500, 1399],
    "uo-180-180-095": [400, 1350],
    "uo-180-180-120": [300, 1099],
    "uo-180-180-180": [400, 1284],
}

In [ ]:
style_options = {
    "uo-050-180-180": {"color": "red", "marker": "+"},
    "uo-060-180-180": {"color": "green", "marker": "x"},
    "uo-070-180-180": {"color": "blue", "marker": "x"},
    "uo-100-180-180": {"color": "white", "marker": "s", "edgecolors": "pink"},
    "uo-145-180-180": {"color": "cyan", "marker": "s"},
    "uo-180-180-070": {"color": "grey", "marker": "^"},
    "uo-180-180-095": {"color": "white", "marker": "^", "edgecolors": "orange"},
    "uo-180-180-120": {
        "color": "black",
        "marker": "o",
    },
    "uo-180-180-180": {"color": "white", "marker": "o", "edgecolors": "purple"},
}

### Load trajectories

In [ ]:
import pathlib

from pedpy import TrajectoryData, TrajectoryUnit, load_trajectory

folder = pathlib.Path("demo-data/uni-directional")
trajectories = {}
trajectories_in_steady_state = {}
for file in folder.glob("uo*.txt"):
    trajectory = load_trajectory(
        trajectory_file=file,
        default_frame_rate=16.0,
        default_unit=TrajectoryUnit.CENTIMETER,
    )
    trajectory_in_steady_state = TrajectoryData(
        data=trajectory.data[trajectory.data.frame.between(steady_states[file.stem][0], steady_states[file.stem][1])],
        frame_rate=trajectory.frame_rate,
    )

    trajectories[file.stem] = trajectory
    trajectories_in_steady_state[file.stem] = trajectory_in_steady_state

### Define measurement setup

In [ ]:
from shapely import Polygon

from pedpy import WalkableArea

walkable_area = WalkableArea(
    Polygon(
        [
            (2.8, -6.5),
            (2.8, -4),
            (1.8, -4),
            (1.8, 4),
            (2.8, 4),
            (2.8, 8),
            (-1, 8),
            (-1, 4),
            (0, 4),
            (0, -4),
            (-1, -4),
            (-1, -6.5),
        ]
    )
)

In [ ]:
from pedpy import MeasurementArea, MeasurementLine

measurement_area = MeasurementArea([(0, -2), (0, 0), (1.8, 0), (1.8, -2)])
measurement_line = MeasurementLine([(0, 0), (1.8, 0)])

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

fig, axs = plt.subplots(3, int(len(trajectories) / 3), figsize=(10, 30))

for (name, trajectory), ax in zip(trajectories.items(), axs.ravel(), strict=False):
    ax.set_title(name[:-4])

    ax = plot_measurement_setup(
        traj=trajectory,
        walkable_area=walkable_area,
        measurement_areas=[measurement_area],
        measurement_lines=[measurement_line],
        axes=ax,
        traj_width=0.2,
        traj_start_marker=".",
        traj_end_marker="x",
        ma_color="g",
        ma_line_color="g",
        ma_alpha=0.2,
        ml_color="b",
    )
    ax.set_aspect("equal")

fig.tight_layout()
plt.show()

In [ ]:
from pedpy.column_identifier import *

## Method A

Method A calculates the mean value of flow and density **over time**.
A reference location $x$ in the corridor is taken and studied over a fixed period of time $\Delta t$, here realized by a {class}`measurement line <geometry.MeasurementLine>`.

```{eval-rst}
.. image:: /images/measurement_line.svg
    :align: center
    :width: 60 %
```

Counting the pedestrians crossing the line over time gives the cumulative $N(t)$ curve, from which the flow is obtained:

```{eval-rst}
.. image:: /images/flow.svg
    :align: center
    :width: 80 %
```

The flow $\langle J \rangle_{\Delta t}$ and the time mean speed $\langle v \rangle_{\Delta t}$ are defined as

$$
    \langle J \rangle_{\Delta t} = \frac{N_{\Delta t}}{t_{N_{\Delta t}}}
    \qquad \text{and} \qquad
    \langle v \rangle_{\Delta t} = \frac{1}{N_{\Delta t}} \sum_{i=1}^{N_{\Delta t}} v_i(t),
$$

where $N_{\Delta t}$ is the number of persons passing the location $x$ during the time interval $\Delta t$.
$t_{N_{\Delta t}}$ is the time between the first and the last of these $N_{\Delta t}$ pedestrians, i.e., the time actually used to pass the location, which can differ from $\Delta t$.

The density cannot be measured at a line directly, so it is derived from the hydrodynamic relation $J = \rho \, v \, b$:

$$
    \rho = \frac{\langle J \rangle_{\Delta t}}{\langle v \rangle_{\Delta t} \cdot b_{cor}},
$$

with $b_{cor}$ the width of the corridor.

Method A is well suited when one is interested in the flow at a specific location, e.g., at bottlenecks or exits.


### Compute n-t and flow

In [ ]:
from pedpy import (
    SpeedCalculation,
    compute_flow,
    compute_individual_speed,
    compute_n_t,
)

nts = {}
flows = {}

for name, trajectory in trajectories.items():
    individual_speed = compute_individual_speed(
        traj_data=trajectories_in_steady_state[name],
        frame_step=10,
        speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
    )

    nt, crossing = compute_n_t(
        traj_data=trajectories_in_steady_state[name],
        measurement_line=measurement_line,
    )

    delta_frame = int(10 * trajectory.frame_rate)
    flow = compute_flow(
        nt=nt,
        crossing_frames=crossing,
        individual_speed=individual_speed,
        delta_frame=delta_frame,
        frame_rate=trajectory.frame_rate,
    )

    flows[name] = flow
    nts[name] = nt

### Plot n-t diagram

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_nt

prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]

fig = plt.figure(figsize=(7, 7))
ax1 = fig.add_subplot(111)
ii = 0

for name, nt in nts.items():
    plot_nt(axes=ax1, nt=nt, label=name, color=colors[ii])
    ii += 1

ax1.legend()
ax1.set_xlabel("time / s")
ax1.set_ylabel("cumulative pedestrians")
plt.show()

### Plot fundamental diagram

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

style_options = {
    "uo-050-180-180": {"color": "red", "marker": "+"},
    "uo-060-180-180": {"color": "limegreen", "marker": "x"},
    "uo-070-180-180": {"color": "blue", "marker": "x"},
    "uo-100-180-180": {
        "color": "white",
        "marker": "s",
        "edgecolors": "hotpink",
    },
    "uo-145-180-180": {"color": "cyan", "marker": "s"},
    "uo-180-180-070": {"color": "darkgrey", "marker": "^"},
    "uo-180-180-095": {
        "color": "white",
        "marker": "^",
        "edgecolors": "darkorange",
    },
    "uo-180-180-120": {
        "color": "black",
        "marker": "o",
    },
    "uo-180-180-180": {
        "color": "white",
        "marker": "o",
        "edgecolors": "mediumorchid",
    },
}

fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2, figsize=(15, 6))

for name, flow in flows.items():
    ax0.scatter(
        flow[FLOW_COL] / (flow[MEAN_SPEED_COL] * measurement_line.length),
        flow[MEAN_SPEED_COL],
        label=name,
        **style_options[name],
    )

ax0.set_xlim(left=0, right=4)
ax0.set_ylim(bottom=0, top=2.5)

ax0.set_xlabel("rho / 1/m^2")
ax0.set_ylabel("v/ m/s")
ax0.grid()
ax0.legend()

img = mpimg.imread(folder / "comparison/method_a_uo.png")
ax1.set_title("paper")
ax1.imshow(img)
ax1.axis("off")

plt.show()

## Method B

Method B measures the mean value of speed and density **over space and time**, so each pedestrian contributes a single data point to the fundamental diagram.
A segment of length $\Delta x$ in the corridor is taken as {class}`measurement area <geometry.MeasurementArea>`.

```{eval-rst}
.. image:: /images/passing_measurement_area.svg
    :align: center
    :width: 60 %
```

The speed $\langle v \rangle_i$ of each person is the length $\Delta x$ of the measurement area divided by the time he or she needs to cross the area:

$$
    \langle v \rangle_i = \frac{\Delta x}{t_{out} - t_{in}},
$$

where $t_{in}$ and $t_{out}$ are the times a person enters and exits the measurement area.
The density $\langle \rho \rangle_i$ for each person is averaged over the same time span:

$$
    \langle \rho \rangle_i = \frac{1}{t_{out} - t_{in}} \int_{t_{in}}^{t_{out}} \frac{N'(t)}{b_{cor} \cdot \Delta x} \, dt,
$$

where $b_{cor}$ is the width of the measurement area and $N'(t)$ the number of persons in this area at time $t$.

Method B is useful when the individual behavior of pedestrians is of interest.


### Compute individual speed and density while passing the measurement area

In [ ]:
from pedpy import (
    compute_classic_density,
    compute_frame_range_in_area,
    compute_passing_density,
    compute_passing_speed,
)

passing_densities = {}
passing_speeds = {}

for name, trajectory in trajectories.items():
    frames_in_area, passing_area = compute_frame_range_in_area(
        traj_data=trajectories_in_steady_state[name],
        measurement_line=measurement_line,
        width=2,
    )

    passing_speed = compute_passing_speed(
        frames_in_area=frames_in_area,
        frame_rate=trajectory.frame_rate,
        distance=2.0,
    )

    density_per_frame = compute_classic_density(
        traj_data=trajectories_in_steady_state[name],
        measurement_area=passing_area,
    )
    passing_density = compute_passing_density(density_per_frame=density_per_frame, frames=frames_in_area)

    passing_speeds[name] = passing_speed
    passing_densities[name] = passing_density

### Plot fundamental diagram

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2, figsize=(15, 6))

for name in passing_speeds.keys():
    ax0.scatter(
        passing_densities[name][DENSITY_COL],
        passing_speeds[name][SPEED_COL],
        label=name,
        **style_options[name],
    )
ax0.set_xlim(left=0, right=4)
ax0.set_ylim(bottom=0, top=2.5)

ax0.set_xlabel("rho / 1/m^2")
ax0.set_ylabel("v/ m/s")
ax0.grid()
ax0.legend()

img = mpimg.imread(folder / "comparison/method_b_uo.png")
ax1.set_title("paper")
ax1.imshow(img)
ax1.axis("off")

plt.show()

## Method C

Method C, also called the **classic method**, measures per frame inside a {class}`measurement area <geometry.MeasurementArea>`.

```{eval-rst}
.. image:: /images/classic_density.svg
    :align: center
    :width: 60 %
```

The density $\langle \rho \rangle_{\Delta x}$ is the number of pedestrians divided by the area of the measurement section:

$$
    \langle \rho \rangle_{\Delta x} = \frac{N}{b_{cor} \cdot \Delta x}
$$

The spatial mean speed is the average of the instantaneous speeds $v_i(t)$ of all $N$ pedestrians in the measurement area at time $t$:

$$
    \langle v \rangle_{\Delta x} = \frac{1}{N} \sum_{i=1}^{N} v_i(t)
$$

Method C is simple and fast to compute, but the density is restricted to multiples of $1/A$, which leads to strong fluctuations and scatter in the resulting fundamental diagram.


### Compute density and speed

In [ ]:
from pedpy import (
    SpeedCalculation,
    compute_classic_density,
    compute_mean_speed_per_frame,
)

classic_densities = {}
mean_area_speeds = {}

for name, trajectory in trajectories.items():
    individual_speed = compute_individual_speed(
        traj_data=trajectories_in_steady_state[name],
        frame_step=5,
        speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
    )
    mean_area_speed = compute_mean_speed_per_frame(
        traj_data=trajectories_in_steady_state[name],
        measurement_area=measurement_area,
        individual_speed=individual_speed,
    )

    classic_density = compute_classic_density(
        traj_data=trajectories_in_steady_state[name],
        measurement_area=measurement_area,
    )

    classic_densities[name] = classic_density
    mean_area_speeds[name] = mean_area_speed

### Plot time series data

In [ ]:
from pedpy import PEDPY_BLUE, PEDPY_RED, plot_density, plot_speed

fig, ax = plt.subplots(nrows=len(trajectories.values()), ncols=2, figsize=(20, 60))
row = 0

for name, trajectory in trajectories.items():
    ax[row, 0].annotate(
        name,
        xy=(0.5, 1),
        xytext=(-ax[row, 0].yaxis.labelpad - 5, 0),
        xycoords=ax[row, 0].yaxis.label,
        textcoords="offset points",
        size="xx-large",
        ha="right",
        va="center",
        rotation=90,
    )

    plot_density(axes=ax[row, 0], density=classic_densities[name], color=PEDPY_BLUE)
    ax[row, 0].set_xlim(left=0)
    ax[row, 0].set_ylim(bottom=0, top=4)
    ax[row, 0].grid()

    plot_speed(axes=ax[row, 1], speed=mean_area_speeds[name], color=PEDPY_RED)
    ax[row, 1].set_xlim(
        left=0,
    )
    ax[row, 1].set_ylim(bottom=0, top=3)
    ax[row, 1].grid()

    row += 1
ax[0, 1].set_title("Speed", size="xx-large")
ax[0, 0].set_title("Density", size="xx-large")

plt.show()

### Plot fundamental diagram

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2, figsize=(15, 6))

ax0.set_title("Fundamental Diagram")

for name in classic_densities.keys():
    ax0.scatter(
        classic_densities[name][DENSITY_COL],
        mean_area_speeds[name][SPEED_COL],
        alpha=1,
        label=name,
        **style_options[name],
    )
ax0.set_xlim(left=0, right=4)
ax0.set_ylim(bottom=0, top=2.5)

ax0.set_xlabel("rho / 1/m^2")
ax0.set_ylabel("v/ m/s")
ax0.grid()
ax0.legend()

img = mpimg.imread(folder / "comparison/method_c_uo.png")
ax1.set_title("paper")
ax1.imshow(img)
ax1.axis("off")
plt.show()

## Method D

Method D is based on **Voronoi diagrams**, a decomposition of a metric space determined by distances to a specified discrete set of objects.
At any time the positions of the pedestrians can be represented as a set of points, from which the Voronoi diagram is generated and the Voronoi cell area $A_i$ of each person $i$ is obtained.

```{eval-rst}
.. image:: /images/voronoi_density.svg
    :align: center
    :width: 60 %
```

The density and speed distribution of the space, $\rho_{xy}$ and $v_{xy}$, are then defined as

$$
    \rho_{xy} = 1/A_i \quad \text{and} \quad v_{xy} = v_i(t) \qquad \text{if } (x,y) \in A_i,
$$

where $v_i(t)$ is the instantaneous speed of each person.
The Voronoi density and speed for the measurement area follow by integrating over the area:

$$
    \langle \rho \rangle_v = \frac{\iint \rho_{xy} \, dx \, dy}{b_{cor} \cdot \Delta x}
    \qquad \text{and} \qquad
    \langle v \rangle_v = \frac{\iint v_{xy} \, dx \, dy}{b_{cor} \cdot \Delta x}
$$

Method D leads to much smaller fluctuations than Method C and allows measurements on small areas, which makes it the preferred method for most analyses.
In situations with low densities the Voronoi cells can become very large, which can be avoided by restricting them with a cutoff radius, both variants are shown below.


### Compute density and speed

#### Without cutoff radius

The Voronoi cells are only limited by the geometry, so in sparse situations they can grow very large.

```{eval-rst}
.. image:: /images/voronoi_wo_cutoff_wo_bounds.svg
    :align: center
    :width: 60 %
```

In [ ]:
from pedpy import (
    SpeedCalculation,
    compute_individual_voronoi_polygons,
    compute_voronoi_density,
    compute_voronoi_speed,
)

voronoi_densities = {}
voronoi_speeds = {}
individual_speeds = {}
individuals = {}

for name, trajectory in trajectories.items():
    individual = compute_individual_voronoi_polygons(
        traj_data=trajectories_in_steady_state[name],
        walkable_area=walkable_area,
    )
    voronoi_density, intersecting = compute_voronoi_density(
        individual_voronoi_data=individual,
        measurement_area=measurement_area,
    )

    individual_speed = compute_individual_speed(
        traj_data=trajectories_in_steady_state[name],
        frame_step=5,
        speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
    )
    voronoi_speed = compute_voronoi_speed(
        traj_data=trajectories_in_steady_state[name],
        individual_voronoi_intersection=intersecting,
        measurement_area=measurement_area,
        individual_speed=individual_speed,
    )

    voronoi_densities[name] = voronoi_density
    voronoi_speeds[name] = voronoi_speed
    individual_speeds[name] = individual_speed
    individuals[name] = intersecting

#### With cutoff radius

Each Voronoi cell is additionally intersected with a disk of the given cutoff radius around the pedestrian, which bounds the area a single person can occupy.

```{eval-rst}
.. image:: /images/voronoi_w_cutoff_wo_bounds.svg
    :align: center
    :width: 60 %
```

In [ ]:
from pedpy import (
    Cutoff,
    SpeedCalculation,
    compute_voronoi_density,
    compute_voronoi_speed,
)

voronoi_densities_cutoff = {}
voronoi_speeds_cutoff = {}
individual_speeds_cutoff = {}
individual_cutoffs = {}
for name, trajectory in trajectories.items():
    individual_cutoff = compute_individual_voronoi_polygons(
        traj_data=trajectories_in_steady_state[name],
        walkable_area=walkable_area,
        cut_off=Cutoff(radius=0.8, quad_segments=3),
    )
    voronoi_density_cutoff, intersecting_cutoff = compute_voronoi_density(
        individual_voronoi_data=individual_cutoff,
        measurement_area=measurement_area,
    )

    individual_speed = compute_individual_speed(
        traj_data=trajectories_in_steady_state[name],
        frame_step=5,
        speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
    )

    voronoi_speed_cutoff = compute_voronoi_speed(
        traj_data=trajectories_in_steady_state[name],
        individual_voronoi_intersection=intersecting_cutoff,
        individual_speed=individual_speed,
        measurement_area=measurement_area,
    )

    voronoi_densities_cutoff[name] = voronoi_density_cutoff
    voronoi_speeds_cutoff[name] = voronoi_speed_cutoff
    individual_speeds_cutoff[name] = individual_speed
    individual_cutoffs[name] = intersecting_cutoff

### Plot time-series data

In [ ]:
from pedpy import PEDPY_BLUE, PEDPY_ORANGE, plot_density, plot_speed

fig, ax = plt.subplots(nrows=len(trajectories.values()), ncols=2, figsize=(20, 60))
row = 0

for name, trajectory in trajectories.items():
    ax[row, 0].annotate(
        name,
        xy=(0.5, 1),
        xytext=(-ax[row, 0].yaxis.labelpad - 5, 0),
        xycoords=ax[row, 0].yaxis.label,
        textcoords="offset points",
        size="xx-large",
        ha="right",
        va="center",
        rotation=90,
    )

    plot_density(axes=ax[row, 0], density=voronoi_densities[name], label="without cut-off", color=PEDPY_BLUE)
    plot_density(axes=ax[row, 0], density=voronoi_densities_cutoff[name], label="with cut-off", color=PEDPY_ORANGE)
    ax[row, 0].set_xlim(left=0)
    ax[row, 0].set_ylim(bottom=0, top=4)
    ax[row, 0].grid()
    ax[row, 0].legend()

    plot_speed(axes=ax[row, 1], speed=voronoi_speeds[name], label="without cut-off", color=PEDPY_BLUE)
    plot_speed(axes=ax[row, 1], speed=voronoi_speeds_cutoff[name], label="with cut-off", color=PEDPY_ORANGE)
    ax[row, 1].set_xlim(
        left=0,
    )
    ax[row, 1].set_ylim(bottom=0, top=3)
    ax[row, 1].grid()
    ax[row, 1].legend()
    row += 1

ax[0, 1].set_title("Speed", size="xx-large")
ax[0, 0].set_title("Density", size="xx-large")
plt.show()

### Plot fundamental diagram

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

fig, (ax0, ax1, ax2) = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
fig.suptitle("Fundamental diagram")

ax0.set_title("without cutoff")
for name in voronoi_densities.keys():
    ax0.scatter(
        voronoi_densities[name][DENSITY_COL],
        voronoi_speeds[name][SPEED_COL],
        alpha=1,
        label=name,
        **style_options[name],
    )
ax0.set_xlim(left=0, right=4)
ax0.set_ylim(bottom=0, top=2.5)

ax0.set_xlabel("rho / 1/m^2")
ax0.set_ylabel("v/ m/s")
ax0.grid()
ax0.legend()

ax1.set_title("with cutoff")
for name in voronoi_densities_cutoff.keys():
    ax1.scatter(
        voronoi_densities_cutoff[name][DENSITY_COL],
        voronoi_speeds_cutoff[name][SPEED_COL],
        alpha=1,
        label=name,
        **style_options[name],
    )
ax1.set_xlim(left=0, right=4)
ax1.set_ylim(bottom=0, top=2.5)

ax1.set_xlabel("rho / 1/m^2")
ax1.set_ylabel("v/ m/s")
ax1.grid()
ax1.legend()

img = mpimg.imread(folder / "comparison/method_d_uo.png")
ax2.set_title("paper")
ax2.imshow(img)
ax2.axis("off")
plt.show()